# __*HADOOP INSTALLATION__

In [ ]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz

--2026-03-24 10:20:50--  https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 730107476 (696M) [application/x-gzip]
Saving to: ‘hadoop-3.3.6.tar.gz’

hadoop-3.3.6.tar.gz 100%[===================>] 696.28M  18.7MB/s    in 38s     

2026-03-24 10:21:29 (18.2 MB/s) - ‘hadoop-3.3.6.tar.gz’ saved [730107476/730107476]



In [ ]:
import os

os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += ":/content/hadoop-3.3.6/bin"

In [ ]:
%%bash
cat <<EOF > hadoop-3.3.6/etc/hadoop/core-site.xml
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
EOF

In [ ]:
%%bash
cat <<EOF > hadoop-3.3.6/etc/hadoop/hdfs-site.xml
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
</configuration>
EOF
readlink -f /usr/bin/java

/usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [ ]:
%%bash
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$PATH:$JAVA_HOME/bin
echo $JAVA_HOME
echo 'export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64' >> hadoop-3.3.6/etc/hadoop/hadoop-env.sh
hadoop namenode -format

/usr/lib/jvm/java-17-openjdk-amd64



2026-03-24 10:21:47,658 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = 8b547508de17/172.28.0.12
STARTUP_MSG:   args = [-format]
STARTUP_MSG:   version = 3.3.6
STARTUP_MSG:   classpath = /content/hadoop-3.3.6/etc/hadoop:/content/hadoop-3.3.6/share/hadoop/common/lib/snappy-java-1.1.8.2.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/jaxb-impl-2.2.3-1.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-resolver-dns-native-macos-4.1.89.Final-osx-x86_64.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-transport-udt-4.1.89.Final.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-handler-4.1.89.Final.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/curator-framework-5.2.0.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/netty-buffer-4.1.89.Final.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/kerb-core-1.0.1.jar:/content/hadoop-3.3.6/share/hadoop/common/lib/reload4j

In [ ]:
%%bash
cat <<EOF >> hadoop-3.3.6/etc/hadoop/hadoop-env.sh
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
EOF

In [ ]:
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!hdfs --daemon start secondarynamenode

In [ ]:
!jps
!hdfs dfs -mkdir /test
!hdfs dfs -ls /

3811 Jps
3767 SecondaryNameNode
3705 DataNode
3627 NameNode
Found 1 items
drwxr-xr-x   - root supergroup          0 2026-03-24 10:22 /test


# __BÀI 1__

In [ ]:
!mkdir -p assignment1

In [ ]:
%%bash
cat <<EOF > assignment1/Assignment1.java
import java.io.IOException;
import java.util.Map;
import java.util.TreeMap;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.*;
import org.apache.hadoop.mapreduce.lib.output.*;

public class Assignment1 {

    public static class TitleMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
                throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 2) {
                String movieId = parts[0].trim();
                String title = parts[1].trim();

                context.write(new Text(movieId), new Text("T|" + title));
            }
        }
    }

    public static class RatingMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
                throws IOException, InterruptedException {

            // UserID,MovieID,Rating,Timestamp
            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String movieId = parts[1].trim();
                String rating = parts[2].trim();

                context.write(new Text(movieId), new Text("R|" + rating));
            }
        }
    }

    public static class ReducerClass extends Reducer<Text, Text, Text, Text> {

        private Map<String, String> resultMap = new TreeMap<>();
        private String maxMovie = "";
        private double maxRating = Double.MIN_VALUE;

        public void reduce(Text key, Iterable<Text> values, Context context)
                throws IOException, InterruptedException {

            String title = "";
            double sum = 0;
            int count = 0;

            for (Text val : values) {
                String v = val.toString();

                if (v.startsWith("T|")) {
                    title = v.substring(2);
                } else if (v.startsWith("R|")) {
                    double rating = Double.parseDouble(v.substring(2).trim());
                    sum += rating;
                    count++;
                }
            }

            if (!title.isEmpty() && count > 0) {
                double avg = sum / count;

                String info = String.format("Average rating: %.2f (Total ratings: %d)", avg, count);

                resultMap.put(title, info);

                if (count >= 5 && avg > maxRating) {
                    maxRating = avg;
                    maxMovie = title;
                }
            }
        }
        @Override
        protected void cleanup(Context context)
                throws IOException, InterruptedException {

            for (Map.Entry<String, String> entry : resultMap.entrySet()) {
                context.write(new Text(entry.getKey()), new Text(entry.getValue()));
            }

            if (!maxMovie.isEmpty()) {
                context.write(new Text("RESULT"),
                    new Text(maxMovie + " is the highest rated movie with an average rating of "
                            + maxRating + " among movies with at least 5 ratings."));
            }
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "Movie Rating Join");

        job.setJarByClass(Assignment1.class);

        // Multiple input
        MultipleInputs.addInputPath(job, new Path(args[0] + "/movies.txt"),
                TextInputFormat.class, TitleMapper.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/ratings_1.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/ratings_2.txt"),
                TextInputFormat.class, RatingMapper.class);

        job.setReducerClass(ReducerClass.class);
        job.setNumReduceTasks(1);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [ ]:
!javac -classpath `hadoop classpath` -d assignment1 assignment1/Assignment1.java
!jar -cvf assignment1.jar -C assignment1/ .

added manifest
adding: Assignment1$RatingMapper.class(in = 2071) (out= 844)(deflated 59%)
adding: Assignment1$ReducerClass.class(in = 3820) (out= 1751)(deflated 54%)
adding: Assignment1$TitleMapper.class(in = 2069) (out= 843)(deflated 59%)
adding: Assignment1.class(in = 2216) (out= 1083)(deflated 51%)
adding: Assignment1.java(in = 4269) (out= 1193)(deflated 72%)


In [ ]:
!hdfs dfs -mkdir -p /assignment1
!hdfs dfs -put -f movies.txt /assignment1/
!hdfs dfs -put -f ratings_1.txt /assignment1/
!hdfs dfs -put -f ratings_2.txt /assignment1/

In [ ]:
!hdfs dfs -rm -r -f /assignment1/output
!hadoop jar assignment1.jar Assignment1 /assignment1 /assignment1/output

2026-03-24 10:27:43,591 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-24 10:27:43,751 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-24 10:27:43,751 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-24 10:27:43,999 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-24 10:27:44,177 INFO input.FileInputFormat: Total input files to process : 1
2026-03-24 10:27:44,330 INFO input.FileInputFormat: Total input files to process : 2
2026-03-24 10:27:44,369 INFO mapreduce.JobSubmitter: number of splits:3
2026-03-24 10:27:44,735 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local2058348847_0001
2026-03-24 10:27:44,735 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-24 10:27:45,022 INFO mapreduce.Job: The url to track the job: http://localhos

In [ ]:
!hdfs dfs -cat /assignment1/output/part-r-*

American Beauty (1999)	Average rating: 3.75 (Total ratings: 2)
Avatar (2009)	Average rating: 4.75 (Total ratings: 2)
Back to the Future (1985)	Average rating: 4.00 (Total ratings: 2)
Birdman (2014)	Average rating: 3.00 (Total ratings: 2)
Braveheart (1995)	Average rating: 4.25 (Total ratings: 2)
Coco (2017)	Average rating: 4.25 (Total ratings: 2)
Dunkirk (2017)	Average rating: 3.50 (Total ratings: 2)
Fight Club (1999)	Average rating: 4.25 (Total ratings: 2)
Forrest Gump (1994)	Average rating: 3.75 (Total ratings: 2)
Gladiator (2000)	Average rating: 3.25 (Total ratings: 2)
Good Will Hunting (1997)	Average rating: 3.50 (Total ratings: 2)
Gravity (2013)	Average rating: 4.25 (Total ratings: 2)
Inception (2010)	Average rating: 3.75 (Total ratings: 2)
Interstellar (2014)	Average rating: 3.50 (Total ratings: 2)
Jurassic Park (1993)	Average rating: 4.00 (Total ratings: 2)
La La Land (2016)	Average rating: 4.00 (Total ratings: 2)
Mad Max: Fury Road (2015)	Average rating: 4.00 (Total ratings: 2)


In [ ]:
!hdfs dfs -getmerge /assignment1/output/ assignment1/assignmentResult1.txt

# __BÀI 2__

In [ ]:
!mkdir -p assignment2

In [ ]:
%%bash
cat <<EOF > assignment2/Assignment2.java
import java.io.IOException;
import java.util.*;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.*;
import org.apache.hadoop.mapreduce.lib.output.*;

public class Assignment2 {

    public static class MovieMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
                throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String movieId = parts[0].trim();
                String genres = parts[2].trim();

                context.write(new Text(movieId), new Text("M|" + genres));
            }
        }
    }

    public static class RatingMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
                throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String movieId = parts[1].trim();
                String rating = parts[2].trim();

                context.write(new Text(movieId), new Text("R|" + rating));
            }
        }
    }

    public static class ReducerClass extends Reducer<Text, Text, Text, Text> {

        private Map<String, Double> sumMap = new TreeMap<>();
        private Map<String, Integer> countMap = new TreeMap<>();

        public void reduce(Text key, Iterable<Text> values, Context context)
                throws IOException, InterruptedException {

            List<Double> ratings = new ArrayList<>();
            String[] genres = null;

            for (Text val : values) {
                String v = val.toString();

                if (v.startsWith("M|")) {
                    genres = v.substring(2).split("\\\|");
                } else if (v.startsWith("R|")) {
                    ratings.add(Double.parseDouble(v.substring(2)));
                }
            }

            if (genres != null && ratings.size() > 0) {
                for (String genre : genres) {
                    genre = genre.trim();

                    for (double r : ratings) {
                        sumMap.put(genre, sumMap.getOrDefault(genre, 0.0) + r);
                        countMap.put(genre, countMap.getOrDefault(genre, 0) + 1);
                    }
                }
            }
        }
        @Override
        protected void cleanup(Context context)
                throws IOException, InterruptedException {

            for (String genre : sumMap.keySet()) {
                double sum = sumMap.get(genre);
                int count = countMap.get(genre);
                double avg = sum / count;

                String result = String.format("Avg: %.2f, Count: %d", avg, count);
                context.write(new Text(genre), new Text(result));
            }
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "Genre Rating Analysis");

        job.setJarByClass(Assignment2.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/movies.txt"),
                TextInputFormat.class, MovieMapper.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/ratings_1.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/ratings_2.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job, new Path(args[0] + "/users.txt"),
                TextInputFormat.class, RatingMapper.class);

        job.setReducerClass(ReducerClass.class);
        job.setNumReduceTasks(1);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [ ]:
!javac -classpath `hadoop classpath` -d assignment2 assignment2/Assignment2.java
!jar -cvf assignment2.jar -C assignment2/ .

added manifest
adding: Assignment2$MovieMapper.class(in = 2069) (out= 845)(deflated 59%)
adding: Assignment2$RatingMapper.class(in = 2071) (out= 845)(deflated 59%)
adding: Assignment2$ReducerClass.class(in = 3595) (out= 1607)(deflated 55%)
adding: Assignment2.class(in = 2275) (out= 1110)(deflated 51%)
adding: Assignment2.java(in = 4127) (out= 1082)(deflated 73%)


In [ ]:
!hdfs dfs -mkdir -p /assignment2
!hdfs dfs -put -f movies.txt /assignment2/
!hdfs dfs -put -f ratings_1.txt /assignment2/
!hdfs dfs -put -f ratings_2.txt /assignment2/
!hdfs dfs -put -f users.txt /assignment2/

In [ ]:
!hdfs dfs -rm -r -f /assignment2/output
!hadoop jar assignment2.jar Assignment2 /assignment2 /assignment2/output

2026-03-24 10:28:20,201 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-24 10:28:20,509 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-24 10:28:20,509 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-24 10:28:20,840 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-24 10:28:21,213 INFO input.FileInputFormat: Total input files to process : 3
2026-03-24 10:28:21,481 INFO input.FileInputFormat: Total input files to process : 1
2026-03-24 10:28:21,553 INFO mapreduce.JobSubmitter: number of splits:4
2026-03-24 10:28:21,907 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1609958500_0001
2026-03-24 10:28:21,907 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-24 10:28:22,173 INFO mapred.LocalJobRunner: OutputCommitter set in config nul

In [ ]:
!hdfs dfs -cat /assignment2/output/part-r-*

Action	Avg: 3.73, Count: 20
Adventure	Avg: 3.85, Count: 30
Animation	Avg: 4.17, Count: 6
Biography	Avg: 4.14, Count: 14
Children	Avg: 4.25, Count: 2
Comedy	Avg: 3.96, Count: 12
Crime	Avg: 3.78, Count: 20
Drama	Avg: 3.81, Count: 76
Family	Avg: 4.25, Count: 2
Fantasy	Avg: 3.95, Count: 10
History	Avg: 4.00, Count: 6
Music	Avg: 3.88, Count: 4
Mystery	Avg: 4.00, Count: 8
Romance	Avg: 3.75, Count: 8
Sci-Fi	Avg: 3.85, Count: 20
Thriller	Avg: 3.88, Count: 16
War	Avg: 4.00, Count: 2


In [ ]:
!hdfs dfs -getmerge /assignment2/output/ assignment2/assignmentResult2.txt

# __BÀI 3__

In [ ]:
!mkdir -p assignment3

In [ ]:
%%bash
cat <<EOF > assignment3/Assignment3.java
import java.io.IOException;
import java.util.*;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.*;
import org.apache.hadoop.mapreduce.lib.output.*;

public class Assignment3 {

    public static class RatingMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String userId = parts[0].trim();
                String movieId = parts[1].trim();
                String rating = parts[2].trim();

                context.write(new Text(userId), new Text("R|" + movieId + "|" + rating));
            }
        }
    }

    public static class UserMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String userId = parts[0].trim();
                String gender = parts[1].trim();

                context.write(new Text(userId), new Text("G|" + gender));
            }
        }
    }

    public static class MovieMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {

            String[] parts = value.toString().split(",");

            if (parts.length >= 3) {
                String movieId = parts[0].trim();
                String movieTitle = parts[1].trim();

                context.write(new Text(movieId), new Text("T|" + movieTitle));
            }
        }
    }

    public static class IdentityMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
                throws IOException, InterruptedException {

            String[] parts = value.toString().split("\t");
            if (parts.length == 2) {
                context.write(new Text(parts[0]), new Text(parts[1]));
            }
        }
    }

    public static class MovieIdGenderRatingReducer extends Reducer<Text, Text, Text, Text> {
        public void reduce(Text key, Iterable<Text> values, Context context)
        throws IOException, InterruptedException {

            String gender = null;
            List<String[]> ratings = new ArrayList<>();

            for (Text val : values) {
                String v = val.toString();

                if (v.startsWith("G|")) {
                    gender = v.substring(2);
                } else if (v.startsWith("R|")) {
                    ratings.add(v.substring(2).split("\\\|"));
                }
            }

            if (gender != null) {
                for (String[] r : ratings) {
                    String movieId = r[0];
                    String rating = r[1];
                    context.write(new Text(movieId),
                            new Text("G|" + gender + "|" + rating));
                }
            }
        }
    }

    public static class FinalReducer extends Reducer<Text, Text, Text, Text> {

        private Map<String, Double> maleSum = new TreeMap<>();
        private Map<String, Integer> maleCount = new TreeMap<>();

        private Map<String, Double> femaleSum = new TreeMap<>();
        private Map<String, Integer> femaleCount = new TreeMap<>();

        Map<String, String> resultMap = new TreeMap<>();
        Map<String, String> movieTitleMap = new HashMap<>();

        public void reduce(Text key, Iterable<Text> values, Context context)
                throws IOException, InterruptedException {

            String movieTitle = null;

            for (Text val : values) {
                String v = val.toString();

                if (v.startsWith("T|")) {
                    movieTitle = v.substring(2);
                } else if (v.startsWith("G|")) {

                    String[] parts = v.split("\\\|");

                    String gender = parts[1];
                    double rating = Double.parseDouble(parts[2]);

                    if (gender.equals("M")) {
                        maleSum.put(key.toString(),
                            maleSum.getOrDefault(key.toString(), 0.0) + rating);
                        maleCount.put(key.toString(),
                            maleCount.getOrDefault(key.toString(), 0) + 1);
                    } else if (gender.equals("F")) {
                        femaleSum.put(key.toString(),
                            femaleSum.getOrDefault(key.toString(), 0.0) + rating);
                        femaleCount.put(key.toString(),
                            femaleCount.getOrDefault(key.toString(), 0) + 1);
                    }
                }
            }

            if (movieTitle != null) {
                movieTitleMap.put(key.toString(), movieTitle);
            }
        }

        @Override
        protected void cleanup(Context context)
                throws IOException, InterruptedException {

            Set<String> movies = new TreeSet<>();
            movies.addAll(maleSum.keySet());
            movies.addAll(femaleSum.keySet());

            for (String movieId : movies) {

                String title = movieTitleMap.getOrDefault(movieId, movieId);

                double mAvg = maleCount.containsKey(movieId)
                        ? maleSum.get(movieId) / maleCount.get(movieId) : 0;

                double fAvg = femaleCount.containsKey(movieId)
                        ? femaleSum.get(movieId) / femaleCount.get(movieId) : 0;

                String result = String.format("Male: %.2f, Female: %.2f", mAvg, fAvg);
                resultMap.put(title, result);
            }

            for (String title : resultMap.keySet()) {
                context.write(new Text(title), new Text(resultMap.get(title)));
            }
        }
    }

    public static void main(String[] args) throws Exception {

        // ===== JOB 1 =====
        Configuration conf1 = new Configuration();
        Job job1 = Job.getInstance(conf1, "Join Ratings and Users");

        job1.setJarByClass(Assignment3.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/ratings_1.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/ratings_2.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/users.txt"),
                TextInputFormat.class, UserMapper.class);

        job1.setReducerClass(MovieIdGenderRatingReducer.class);

        job1.setOutputKeyClass(Text.class);
        job1.setOutputValueClass(Text.class);

        Path tempOutput = new Path(args[0] + "/temp_output");
        FileOutputFormat.setOutputPath(job1, tempOutput);

        if (!job1.waitForCompletion(true)) {
            System.exit(1);
        }

        // ===== JOB 2 =====
        Configuration conf2 = new Configuration();
        Job job2 = Job.getInstance(conf2, "Join Movies and Compute Avg");

        job2.setJarByClass(Assignment3.class);

        // input 1: output job1
        MultipleInputs.addInputPath(job2, tempOutput,
                TextInputFormat.class, IdentityMapper.class); // identity mapper

        // input 2: movies
        MultipleInputs.addInputPath(job2, new Path(args[0] + "/movies.txt"),
                TextInputFormat.class, MovieMapper.class);

        job2.setReducerClass(FinalReducer.class);

        job2.setOutputKeyClass(Text.class);
        job2.setOutputValueClass(Text.class);

        FileOutputFormat.setOutputPath(job2, new Path(args[1]));

        System.exit(job2.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [ ]:
!javac -classpath `hadoop classpath` -d assignment3 assignment3/Assignment3.java
!jar -cvf assignment3.jar -C assignment3/ .

added manifest
adding: Assignment3$FinalReducer.class(in = 4267) (out= 1891)(deflated 55%)
adding: Assignment3$IdentityMapper.class(in = 1605) (out= 645)(deflated 59%)
adding: Assignment3$MovieIdGenderRatingReducer.class(in = 2608) (out= 1149)(deflated 55%)
adding: Assignment3$MovieMapper.class(in = 2069) (out= 843)(deflated 59%)
adding: Assignment3$RatingMapper.class(in = 2106) (out= 859)(deflated 59%)
adding: Assignment3$UserMapper.class(in = 2067) (out= 842)(deflated 59%)
adding: Assignment3.class(in = 2641) (out= 1329)(deflated 49%)
adding: Assignment3.java(in = 7982) (out= 1574)(deflated 80%)


In [ ]:
!hdfs dfs -rm -r -f /assignment3
!hdfs dfs -mkdir -p /assignment3
!hdfs dfs -put -f movies.txt /assignment3/
!hdfs dfs -put -f ratings_1.txt /assignment3/
!hdfs dfs -put -f ratings_2.txt /assignment3/
!hdfs dfs -put -f users.txt /assignment3/

Deleted /assignment3


In [ ]:
!hdfs dfs -rm -r -f /assignment3/output
!hadoop jar assignment3.jar Assignment3 /assignment3 /assignment3/output

2026-03-24 10:28:59,476 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-24 10:28:59,786 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-24 10:28:59,786 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-24 10:29:00,010 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-24 10:29:00,327 INFO input.FileInputFormat: Total input files to process : 1
2026-03-24 10:29:00,549 INFO input.FileInputFormat: Total input files to process : 2
2026-03-24 10:29:00,611 INFO mapreduce.JobSubmitter: number of splits:3
2026-03-24 10:29:01,075 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local220148710_0001
2026-03-24 10:29:01,075 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-24 10:29:01,504 INFO mapreduce.Job: The url to track the job: http://localhost

In [ ]:
!hdfs dfs -cat /assignment3/output/part-r-*

American Beauty (1999)	Male: 3.00, Female: 4.50
Avatar (2009)	Male: 5.00, Female: 4.50
Back to the Future (1985)	Male: 4.00, Female: 4.00
Birdman (2014)	Male: 3.00, Female: 3.00
Braveheart (1995)	Male: 4.50, Female: 4.00
Coco (2017)	Male: 5.00, Female: 3.50
Dunkirk (2017)	Male: 4.00, Female: 3.00
Fight Club (1999)	Male: 5.00, Female: 3.50
Forrest Gump (1994)	Male: 2.50, Female: 5.00
Gladiator (2000)	Male: 3.50, Female: 3.00
Good Will Hunting (1997)	Male: 3.00, Female: 4.00
Gravity (2013)	Male: 4.00, Female: 4.50
Inception (2010)	Male: 3.50, Female: 4.00
Interstellar (2014)	Male: 3.50, Female: 3.50
Jurassic Park (1993)	Male: 3.50, Female: 4.50
La La Land (2016)	Male: 5.00, Female: 3.00
Mad Max: Fury Road (2015)	Male: 4.00, Female: 4.00
Manchester by the Sea (2016)	Male: 3.50, Female: 4.50
Memento (2000)	Male: 3.50, Female: 4.50
Moonlight (2016)	Male: 3.00, Female: 4.00
Parasite (2019)	Male: 4.00, Female: 4.00
Pulp Fiction (1994)	Male: 4.50, Female: 3.50
Saving Private Ryan (1998)	Male: 

In [ ]:
!hdfs dfs -getmerge /assignment3/output/ assignment3/assignmentResult3.txt

# __BÀI 4__

In [ ]:
!mkdir -p assignment4

In [ ]:
%%bash
cat <<EOF > assignment4/Assignment4.java

import java.io.IOException;
import java.util.*;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.fs.FileSystem;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.*;
import org.apache.hadoop.mapreduce.lib.output.*;

public class Assignment4 {

    public static class UserMapper extends Mapper<LongWritable, Text, Text, Text> {

        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {
            String[] parts = value.toString().split(",");

            if (parts.length >= 3){
                String userId = parts[0].trim();
                String userAge = parts[2].trim();

                context.write(new Text(userId), new Text("A," + userAge));
            }
        }
    }

    public static class RatingMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {
            String[] parts = value.toString().split(",");

            if (parts.length >= 3){
                String userId = parts[0].trim();
                String movieId = parts[1].trim();
                String rating = parts[2].trim();

                context.write(new Text(userId), new Text("R," + movieId + "," + rating));
            }
        }
    }

    private static String getAgeGroup(int age) {
        if (age <= 18) return "0-18";
        else if (age <= 35) return "18-35";
        else if (age <= 50) return "35-50";
        else return "50+";
    }

    public static class MiddleReducer extends Reducer<Text, Text, Text, Text> {
        public void reduce(Text key, Iterable<Text> values, Context context)
        throws IOException, InterruptedException {

            String age = null;
            List<String[]> ratings = new ArrayList<>();

            for (Text val : values) {
                String v = val.toString();

                if (v.startsWith("A,")) {
                    age = v.substring(2);
                } else if (v.startsWith("R,")) {
                    ratings.add(v.substring(2).split(","));
                }
            }

            if (age != null) {
                int ageInt = Integer.parseInt(age);
                String group = getAgeGroup(ageInt);
                for (String[] r : ratings) {
                    context.write(new Text(r[0]),
                            new Text("G," + group + "," + r[1]));
                }
            }
        }
    }

    public static class MovieMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException {
            String[] parts = value.toString().split(",");

            if (parts.length >= 3){
                String movieId = parts[0].trim();
                String movieTitle = parts[1].trim();

                context.write(new Text(movieId), new Text("T," + movieTitle));
            }
        }
    }

    public static class IdentityMapper extends Mapper<LongWritable, Text, Text, Text> {
        public void map(LongWritable key, Text value, Context context)
        throws IOException, InterruptedException{
            String[] parts = value.toString().split("\t");
            if (parts.length == 2) {

                context.write(new Text(parts[0]), new Text(parts[1]));
            }
        }
    }

    public static class FinalReducer extends Reducer<Text, Text, Text, Text> {

        private Map<String, Map<String, Double>> sumMap = new TreeMap<>();
        private Map<String, Map<String, Integer>> countMap = new TreeMap<>();
        private Map<String, String> titleMap = new HashMap<>();
        private Map<String, String> info = new TreeMap<>();

        public void reduce(Text key, Iterable<Text> values, Context context)
        throws IOException, InterruptedException {
            String movieId = key.toString();

            for (Text val: values){
                String v = val.toString();
                if (v.startsWith("T,")){
                    titleMap.put(movieId, v.substring(2));
                }
                else if (v.startsWith("G,")){
                    String[] parts = v.split(",");
                    String group = parts[1];
                    double rating = Double.parseDouble(parts[2]);

                    sumMap.putIfAbsent(movieId, new HashMap<>());
                    countMap.putIfAbsent(movieId, new HashMap<>());

                    Map<String, Double> sMap = sumMap.get(movieId);
                    Map<String, Integer> cMap = countMap.get(movieId);

                    sMap.put(group, sMap.getOrDefault(group, 0.0) + rating);
                    cMap.put(group, cMap.getOrDefault(group, 0) + 1);
                }
            }
        }

        @Override
        protected void cleanup(Context context)
                throws IOException, InterruptedException {

            for (String movieId : sumMap.keySet()) {

                String title = titleMap.getOrDefault(movieId, movieId);

                Map<String, Double> sMap = sumMap.get(movieId);
                Map<String, Integer> cMap = countMap.get(movieId);

                String[] groups = {"0-18", "18-35", "35-50", "50+"};

                StringBuilder result = new StringBuilder();

                for (String g : groups) {
                    double avg = 0.0;

                    if (cMap.containsKey(g)) {
                        avg = sMap.get(g) / cMap.get(g);
                        result.append(g).append(": ")
                          .append(String.format("%.2f", avg)).append(", ");
                    }
                    else {
                        result.append(g).append(": ")
                          .append("NA").append(", ");
                    }
                }
                info.put(title, result.toString());
            }
            for (Map.Entry<String, String> entry: info.entrySet()){
                context.write(new Text(entry.getKey()), new Text(entry.getValue()));
            }
        }
    }

    public static void main(String[] args) throws Exception {

        // ===== JOB 1 =====
        Configuration conf1 = new Configuration();

        Job job1 = Job.getInstance(conf1, "Join Ratings and Users");

        job1.setJarByClass(Assignment4.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/ratings_1.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/ratings_2.txt"),
                TextInputFormat.class, RatingMapper.class);

        MultipleInputs.addInputPath(job1, new Path(args[0] + "/users.txt"),
                TextInputFormat.class, UserMapper.class);

        job1.setReducerClass(MiddleReducer.class);

        job1.setOutputKeyClass(Text.class);
        job1.setOutputValueClass(Text.class);

        Path tempOutput = new Path(args[0] + "/temp_output");
        FileSystem fs = FileSystem.get(conf1);

        if (fs.exists(tempOutput)) {
            fs.delete(tempOutput, true); // true = xóa đệ quy
        }

        FileOutputFormat.setOutputPath(job1, tempOutput);

        if (!job1.waitForCompletion(true)) {
            System.exit(1);
        }

        // ===== JOB 2 =====
        Configuration conf2 = new Configuration();
        Job job2 = Job.getInstance(conf2, "Join Movies and Compute Avg");

        job2.setJarByClass(Assignment4.class);

        // input 1: output job1
        MultipleInputs.addInputPath(job2, tempOutput,
                TextInputFormat.class, IdentityMapper.class);

        // input 2: movies
        MultipleInputs.addInputPath(job2, new Path(args[0] + "/movies.txt"),
                TextInputFormat.class, MovieMapper.class);

        job2.setReducerClass(FinalReducer.class);

        job2.setOutputKeyClass(Text.class);
        job2.setOutputValueClass(Text.class);

        FileOutputFormat.setOutputPath(job2, new Path(args[1]));

        System.exit(job2.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [ ]:
!javac -classpath `hadoop classpath` -d assignment4 assignment4/Assignment4.java
!jar -cvf assignment4.jar -C assignment4/ .

added manifest
adding: .assignmentResult4.txt.crc(in = 8) (out= 10)(deflated -25%)
adding: Assignment4$FinalReducer.class(in = 4339) (out= 1975)(deflated 54%)
adding: Assignment4$IdentityMapper.class(in = 1605) (out= 646)(deflated 59%)
adding: Assignment4$MiddleReducer.class(in = 2683) (out= 1188)(deflated 55%)
adding: Assignment4$MovieMapper.class(in = 2069) (out= 845)(deflated 59%)
adding: Assignment4$RatingMapper.class(in = 2106) (out= 859)(deflated 59%)
adding: Assignment4$UserMapper.class(in = 2067) (out= 842)(deflated 59%)
adding: Assignment4.class(in = 3090) (out= 1560)(deflated 49%)
adding: Assignment4.java(in = 8225) (out= 1809)(deflated 78%)
adding: assignmentResult4.txt(in = 0) (out= 0)(stored 0%)


In [ ]:
!hdfs dfs -rm -r -f /assignment4
!hdfs dfs -mkdir -p /assignment4
!hdfs dfs -put -f movies.txt /assignment4/
!hdfs dfs -put -f ratings_1.txt /assignment4/
!hdfs dfs -put -f ratings_2.txt /assignment4/
!hdfs dfs -put -f users.txt /assignment4/

Deleted /assignment4


In [ ]:
!hdfs dfs -rm -r -f /assignment4/output
!hadoop jar assignment4.jar Assignment4 /assignment4 /assignment4/output

2026-03-24 10:38:31,785 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-24 10:38:31,940 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-24 10:38:31,941 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-24 10:38:32,077 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-24 10:38:32,219 INFO input.FileInputFormat: Total input files to process : 1
2026-03-24 10:38:32,331 INFO input.FileInputFormat: Total input files to process : 2
2026-03-24 10:38:32,367 INFO mapreduce.JobSubmitter: number of splits:3
2026-03-24 10:38:32,709 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1628626720_0001
2026-03-24 10:38:32,710 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-24 10:38:33,006 INFO mapred.LocalJobRunner: OutputCommitter set in config nul

In [ ]:
!hdfs dfs -cat /assignment4/output/part-r-*

American Beauty (1999)	0-18: NA, 18-35: 3.75, 35-50: NA, 50+: NA, 
Avatar (2009)	0-18: NA, 18-35: 4.50, 35-50: 5.00, 50+: NA, 
Back to the Future (1985)	0-18: NA, 18-35: 4.00, 35-50: NA, 50+: NA, 
Birdman (2014)	0-18: NA, 18-35: 3.00, 35-50: NA, 50+: NA, 
Braveheart (1995)	0-18: NA, 18-35: 4.00, 35-50: 4.50, 50+: NA, 
Coco (2017)	0-18: NA, 18-35: 4.25, 35-50: NA, 50+: NA, 
Dunkirk (2017)	0-18: NA, 18-35: 3.50, 35-50: NA, 50+: NA, 
Fight Club (1999)	0-18: NA, 18-35: 4.25, 35-50: NA, 50+: NA, 
Forrest Gump (1994)	0-18: NA, 18-35: 3.75, 35-50: NA, 50+: NA, 
Gladiator (2000)	0-18: NA, 18-35: 3.00, 35-50: 3.50, 50+: NA, 
Good Will Hunting (1997)	0-18: NA, 18-35: 3.50, 35-50: NA, 50+: NA, 
Gravity (2013)	0-18: NA, 18-35: 4.50, 35-50: 4.00, 50+: NA, 
Inception (2010)	0-18: NA, 18-35: 3.75, 35-50: NA, 50+: NA, 
Interstellar (2014)	0-18: NA, 18-35: 3.50, 35-50: 3.50, 50+: NA, 
Jurassic Park (1993)	0-18: NA, 18-35: 4.00, 35-50: NA, 50+: NA, 
La La Land (2016)	0-18: NA, 18-35: 3.00, 35-50: 5.00, 

In [ ]:
!hdfs dfs -getmerge /assignment4/output/ assignment4/assignmentResult4.txt